## This is the assignment of the week3 which is

**Synthetic DataSet Generator**

In [ ]:
# First importing the required libraries->
import json
import os
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
# Using openrouter for this task instead of groq, huggingface and google ->
# connecting to the openrouter ->
load_dotenv(override=True)
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY")

# Now some case check ->
if not OPENROUTER_API_KEY:
  print(f"NO API KEY was found")
elif not OPENROUTER_API_KEY.startswith("sk-or-v"):
  print(f"An API KEY WAS FOUND, but it does not starts with sk-proj")
elif OPENROUTER_API_KEY.strip() != OPENROUTER_API_KEY:
  print(f"AN API KEY WAS FOUND but it has some errors")
else:
  print(f"API KEY found and it looks great to use")


In [ ]:
# Calling it ->
openrouter_client = OpenAI(api_key=OPENROUTER_API_KEY)

In [ ]:
# Hugging Face Setup ->
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
  if HF_TOKEN.startswith("hf_") and HF_TOKEN.strip() == HF_TOKEN:
    print(f"Hugging face token was found")
  else:
    print(f"Hugging face token format is not correct")
else:
  print("No hugging face token found.")

In [ ]:
HF_MODELS = {
    "Phi-3 Mini": "microsoft/Phi-3-mini-4k-instruct",
    "TinyLlama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}

In [ ]:
loaded_models = {}
def load_hf_model(model_key):
  if model_key in loaded_models:
    return loaded_models[model_key]


  model_name = HF_MODELS[model_key]
  print(f"Loading hugging face model : {model_name}")

  tokenizer = AutoTokenizer.from_pretrained(model_name,token=HF_TOKEN)
  model = AutoModelForCausalLM.from_pretrained(model_name,token=HF_TOKEN,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,device_map="auto")
  loaded_models[model_key] = (tokenizer,model)
  return tokenizer,model


In [ ]:
# Building the prompt only for the json format ->

def build_prompt(ind,purpose,exm_data,sample_size):
  return f"""
  You are an expert synthetic dataset generator.

  Industry: {ind}
  Purpose: {purpose}

  Here is example data showing structure :
  {exm_data}

  Generate {sample_size} new diverse records following the same structure.

  Return ONLY a valid JSON array.
  The output must starts with "[" and ends with the  "]".
  Do not include backticks.
  Do not include explanations.
  Do not include markdown.
  Do not include comments.
"""

In [ ]:
# the generation for the openrouter ->
def generate_with_openrouter(prompt,model_name):
  if not openrouter_client:
    raise ValueError("OpenRouter API key not configured.")

  response = openrouter_client.chat.completions.create(model=model_name,messages=[{"role":"user","content":prompt}],temperature=0.5)
  return response.choices[0].message.content.strip()

In [ ]:
def generate_with_huggingface(prompt, model_key):
    tokenizer, model = load_hf_model(model_key)

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=1200,
        temperature=0.7,
        do_sample=True
    )

    generated_text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated_text.strip()

In [ ]:
# validating the json ->
def validate_json(output_text):
  try:
    data = json.loads(output_text)
    return data, None
  except Exception as e:
    return None, str(e)

In [ ]:
# generating the dataset ->
def generate_dataset(industry,purpose,example_data,model_source,model_choice,sample_size):
  prompt = build_prompt(industry,purpose,example_data,sample_size)
  try:
      if model_source == "OpenRouter":
        raw_output = generate_with_openrouter(prompt,model_choice)
      else:
        raw_output = generate_with_huggingface(prompt,model_choice)
      data,error = validate_json(raw_output)
      if error:
        return f"JSON Error: {error}", raw_output

      return "Generated successfully",json.dumps(data,indent=2)
  except Exception as e:
    return f"Error: {str(e)}",""


In [ ]:
# Gradio generation ->
with gr.Blocks() as demo:
  gr.Markdown("## Synthetic DataSet Generator (JSON ONLY)")

  industry = gr.Textbox(label="Industry")
  purpose = gr.Textbox(label="Data Purpose")
  example_data = gr.Textbox(label="Example Data (JSON),lines=8")

  model_source = gr.Radio(["OpenRouter","Hugging face"],value="Hugging face",label="Model Source")

  model_choice = gr.Dropdown(choices=["Phi-3 Mini","TinyLlama","llama3.2:latest"],value="Phi-3 Mini",label="Select Model")
  sample_size = gr.Slider(1,50,value=5,label="Sample Size")
  status = gr.Textbox(label="Status")
  output_display = gr.Textbox(label="Generate JSON output",lines=20)
  generate_btn = gr.Button("Generate dataset")

  generate_btn.click(
    generate_dataset,
    inputs=[industry, purpose, example_data, model_source, model_choice, sample_size],
    outputs=[status, output_display]
)

demo.launch()
